# 第7章 读懂 Roofline 图

**操作手册** | 看懂参考线、生成工作点并选择排查方向

本手册对应文档：`docs/part1-profiling/chapter7/index.md`  
本手册对应代码：`code/part1-profiling/chapter7/`

---

### 本章导读

> 第 5 章教你把时间量准，第 6 章教你找到慢在哪个 kernel。本章再往前走一步：把时间、数据量和硬件上限放到同一张 Roofline 图上。
>
> 读完后，你应该能判断一个工作点位于拐点哪一侧、离对应上限还有多远，知道下一步该先查访存还是计算，并把这次结果记成一页以后还能看懂的性能记录。

## Goal

学会读懂 Roofline 图，判断工作点位置并选择排查方向。具体目标：

1. 理解 Roofline 的三个读图动作：横轴位置、纵轴高度、离上限距离
2. 计算算术强度（Arithmetic Intensity, AI）
3. 把第6章的实测数据画到 Roofline 图上
4. 理解 memory ceiling 和 compute ceiling 的来源（source attribution）
5. 根据工作点位置选择下一步排查方向

## Prerequisite

- 已完成第5章 benchmark 与可信计时
- 已完成第6章 用 rocprof 找到慢在哪里
- 理解第2-4章的 Roofline 心智模型
- Python 环境已安装 matplotlib 和 numpy

## Platform

本手册基于以下环境验证：

- **GPU**: AMD Radeon RX 9070 XT (gfx1201)
- **ROCm**: 7.13
- **OS**: Ubuntu 24.04 (native, kernel 6.17.0-35-generic)
- **Python**: 3.10+
- **matplotlib**: 3.5+

其他 RDNA3/RDNA4 架构（gfx1100, gfx1151, gfx1201）均可运行，硬件峰值会有差异。

## Parameter

### Roofline 参考线（source attribution）

以下参数来自第3章独立测得的硬件基线（9070XT + ROCm 7.13）：

| 参数 | 值 | 来源 |
|------|----|----- |
| **memory ceiling** | 510 GB/s | 大数组 copy 的 GDDR6 稳态带宽（第3章实测） |
| **compute ceiling (fp32)** | 10.6 TFLOPS | torch.matmul fp32 实测（第3章实测） |

**重要**: 不要使用未经实测的理论峰值或其他架构的数字。

### 工作点数据（第6章实测）

| 版本 | AI (FLOP/Byte) | 时间 (ms) | 有效带宽 (GB/s) |
|------|----------------|-----------|----------------|
| coalesced | 0.083 | 0.334 | 603 |
| linecross stride=32 | 0.083 | 2.25 | 89.7 |

## 7.1 Roofline 只看三件事

这一节把 Roofline 压缩成三个读图动作，不重新推导第 2–4 章已经讲过的公式。把一次实测结果画到图上得到的那个点，后面统一叫**工作点**。

1. **看横轴落在哪一侧**：算术强度（Arithmetic Intensity，AI）表示每搬运 1 Byte 数据做多少次计算。斜线和水平线的交点叫拐点；工作点在拐点左侧，理论上更容易受带宽限制，在右侧则更容易受算力限制。
2. **看纵轴有多高**：纵轴是实际计算性能，通常用 FLOPS 表示。工作点越高，说明单位时间完成的计算越多。
3. **看它离对应上限还有多远**：左侧工作点主要看它到带宽斜线的垂直差距，右侧工作点主要看它到计算水平线的差距。差距很大时，再继续查访存、计算路径、launch 或同步。

| 工作点位置 | 先想到什么 | 常见下一步 |
| ---- | ---- | ---- |
| AI 小，接近带宽斜线 | 典型 memory-bound | 减少访存、做融合或提高数据复用 |
| AI 小，离斜线很远 | 访存效率不高 | 检查地址是否连续、是否有多余读写 |
| AI 大，靠近水平线 | 典型 compute-bound | 使用 WMMA、降精度或减少计算 |
| 离两条线都远 | 还有别的开销 | 检查 launch、同步和输入规模 |

Roofline 不会直接告诉你哪一行代码有问题。它更像一张地图：先根据 AI 选择访存或计算方向，再看工作点离相应上限还有多少空间。

## 7.2 把 vector add 放到图上

这一节用第 6 章的 vector add 走一遍完整计算。

每个 float32 元素需要：

```text
读 a[i]：4 Byte
读 b[i]：4 Byte
写 c[i]：4 Byte
做加法：1 FLOP

算术强度 AI = 1 / 12 ≈ 0.083 FLOP/Byte
```

`0.083 FLOP/Byte` 很低，所以 vector add 会落在图的左侧，属于典型的 memory-bound 算子。

画工作点只需要三样东西：

| 信息 | 从哪里来 |
| ---- | ---- |
| 计算量 F | 从 kernel 代码数操作数 |
| 数据量 B | 从输入、输出的数据类型和元素个数计算 |
| 时间 t | 用第 5 章的 GPU event 实测 |

计算关系是：

```text
算术强度 AI = F / B
实际性能 P  = F / t
有效带宽    = B / t
```

## Execution

### 步骤1：定位仓库根目录并加载第6章结果

In [ ]:
import os
import subprocess
from pathlib import Path

# 定位仓库根目录
REPO_ROOT = Path.cwd().resolve().parents[1] if "notebooks" in str(Path.cwd()) else Path.cwd()
WORK_DIR = REPO_ROOT / "code/part1-profiling/chapter7"
os.chdir(WORK_DIR)

print(f"仓库根目录: {REPO_ROOT}")
print(f"工作目录: {WORK_DIR}")
print(f"当前目录: {Path.cwd()}")

### 步骤2：计算算术强度（arithmetic intensity, inline 示例）

演示如何从 kernel 代码计算 AI（arithmetic intensity，算术强度）：

对于 `c[i] = a[i] + b[i]`，每个 float32 元素的搬运量为 $4 + 4 + 4 = 12$ Byte，计算量为 $1$ FLOP，因此：

$$
AI = \frac{F}{B} = \frac{1\ \text{FLOP}}{12\ \text{Byte}} \approx 0.083\ \text{FLOP/Byte}
$$

这个值远低于 Roofline 拐点（约 20.78 FLOP/Byte），说明 vector add 理论上处于 memory-bound 区域。

In [ ]:
# vector add: c[i] = a[i] + b[i]
# 每个元素（float32）：
#   - 读 a[i]: 4 Byte
#   - 读 b[i]: 4 Byte
#   - 写 c[i]: 4 Byte
#   - 做加法: 1 FLOP

bytes_per_elem = 4 + 4 + 4  # 读 a, 读 b, 写 c
flops_per_elem = 1          # 一次加法

AI = flops_per_elem / bytes_per_elem
print(f"vector add 算术强度 (arithmetic intensity):")
print(f"  bytes_per_elem = {bytes_per_elem} Byte")
print(f"  flops_per_elem = {flops_per_elem} FLOP")
print(f"  AI = {AI:.4f} FLOP/Byte")
print(f"\nAI 很低（{AI:.3f}），说明 vector add 是典型的 memory-bound 算子")

### 步骤3：加载第6章实测数据

下面用 Radeon RX 9070 XT（gfx1201）+ ROCm 7.13 + 原生 Ubuntu 24.04 的这组结果演示：

| 版本 | AI | 时间 | 实际性能 | 有效带宽 |
| ---- | ----: | ----: | ----: | ----: |
| coalesced | 0.083 FLOP/Byte | 0.334 ms | 0.0503 TFLOPS | 603 GB/s |
| linecross stride=32 | 0.083 FLOP/Byte | 2.25 ms | 0.00748 TFLOPS | 89.7 GB/s |

> **注意**：coalesced 按 `12 × n / t` 换算出的有效带宽是 603 GB/s，高于 510 GB/s 的 GDDR6 参考线。这不表示显存突破了硬件上限；有效带宽统计的是算法有效字节，而本例的工作集和写路径还可能受到 cache 等因素影响。这个点更适合用来比较两个实现，而不是当作实际 DRAM 流量。

In [ ]:
# 第6章实测数据（9070XT + ROCm 7.13）
chapter6_results = {
    "coalesced": {
        "time_ms": 0.334,
        "effective_bw_gbps": 603,
        "ai": 1.0 / 12,  # ≈ 0.083 FLOP/Byte
    },
    "linecross_stride32": {
        "time_ms": 2.25,
        "effective_bw_gbps": 89.7,
        "ai": 1.0 / 12,  # ≈ 0.083 FLOP/Byte
    }
}

# 计算实际性能（TFLOPS）
for name, data in chapter6_results.items():
    # P = AI × BW (GB/s -> TFLOPS)
    data["performance_tflops"] = data["ai"] * data["effective_bw_gbps"] / 1e3

print("第6章实测数据:")
for name, data in chapter6_results.items():
    print(f"\n{name}:")
    print(f"  AI = {data['ai']:.4f} FLOP/Byte")
    print(f"  time = {data['time_ms']:.3f} ms")
    print(f"  effective_bw = {data['effective_bw_gbps']:.1f} GB/s")
    print(f"  performance = {data['performance_tflops']:.4f} TFLOPS")

### 步骤4：定义硬件参考线（source attribution）

Roofline 的参考线沿用第 3 章独立测得的硬件基线：大数组 copy 的 GDDR6 稳态带宽为 510 GB/s，fp32 matmul 为 10.6 TFLOPS。

明确标注每条参考线的来源：

In [ ]:
# 硬件参考线（source attribution）
hardware_ceilings = {
    "memory_ceiling": {
        "value_gbps": 510,
        "source": "第3章大数组 copy 实测 GDDR6 稳态带宽",
        "platform": "9070XT (gfx1201) + ROCm 7.13"
    },
    "compute_ceiling_fp32": {
        "value_tflops": 10.6,
        "source": "第3章 torch.matmul fp32 实测",
        "platform": "9070XT (gfx1201) + ROCm 7.13"
    }
}

print("硬件参考线（source attribution）:")
print(f"\nmemory ceiling:")
print(f"  value = {hardware_ceilings['memory_ceiling']['value_gbps']} GB/s")
print(f"  source = {hardware_ceilings['memory_ceiling']['source']}")
print(f"  platform = {hardware_ceilings['memory_ceiling']['platform']}")

print(f"\ncompute ceiling (fp32):")
print(f"  value = {hardware_ceilings['compute_ceiling_fp32']['value_tflops']} TFLOPS")
print(f"  source = {hardware_ceilings['compute_ceiling_fp32']['source']}")
print(f"  platform = {hardware_ceilings['compute_ceiling_fp32']['platform']}")

### 步骤5：运行绘图脚本（plot_roofline_ch6.py）

Roofline 图由仓库中的 `plot_roofline_ch6.py` 生成，脚本使用两组已经测得的数据：

- 第 3 章的大数组 copy 带宽 `510 GB/s` 和 fp32 matmul 性能 `10.6 TFLOPS`，用来画两条参考线；
- 第 6 章的 coalesced、linecross 有效带宽，结合 `AI = 1 / 12` 算出两个工作点的纵坐标。

*（图示：从代码、实测时间和硬件上限得到 Roofline 工作点。先把数据量和计算量算清楚，再把实测时间代进去即可。）*

如果脚本存在，直接运行生成 Roofline 图：

In [ ]:
plot_script = WORK_DIR / "plot_roofline_ch6.py"
output_png = WORK_DIR / "roofline-ch6.png"

if plot_script.exists():
    print(f"运行绘图脚本: {plot_script.name}")
    result = subprocess.run(
        ["python", str(plot_script), "--save"],
        cwd=WORK_DIR,
        capture_output=True,
        text=True,
        timeout=30
    )
    print(result.stdout)
    if result.returncode != 0:
        print(f"错误: {result.stderr}")
    
    if output_png.exists():
        print(f"\n图片已生成: {output_png}")
        # 在 notebook 中显示图片（如果存在）
        from IPython.display import Image, display
        display(Image(filename=str(output_png)))
    else:
        print(f"未找到输出图片: {output_png}")
else:
    print(f"未找到绘图脚本: {plot_script}")

### 步骤6：inline Roofline 绘图（如果脚本不存在）

直接在 notebook 中绘制简化版 Roofline 图：

> **读图要点**：横轴和纵轴都是对数轴，同一格表示倍数变化，而不是固定差值；本例是 fp32，所以只保留 fp32 计算参考线。两个版本按算法口径计算出的算术强度相同（AI ≈ 0.083 FLOP/Byte），区别仅在纵轴——coalesced 的有效带宽为 603 GB/s，点略高于 GDDR6 参考线；linecross stride=32 的有效带宽为 89.7 GB/s，点明显更低。

In [ ]:
try:
    import matplotlib.pyplot as plt
    import numpy as np
    
    # 硬件参数
    BW_GDDR6 = 510  # GB/s
    P_FP32 = 10.6   # TFLOPS
    
    # 工作点
    AI_VADD = 1.0 / 12
    P_COALESCED = chapter6_results["coalesced"]["performance_tflops"]
    P_LINECROSS = chapter6_results["linecross_stride32"]["performance_tflops"]
    
    # 绘图
    fig, ax = plt.subplots(figsize=(9, 5.5), dpi=100)
    
    ai = np.logspace(-2, 3.2, 500)
    
    # memory ceiling (斜线)
    ax.plot(ai, (BW_GDDR6 / 1e3) * ai, "-", color="#2563eb", lw=1.8,
            label=f"memory ceiling = {BW_GDDR6} GB/s (measured)")
    
    # compute ceiling (水平线)
    ax.axhline(P_FP32, color="#dc2626", lw=1.8, ls="--",
               label=f"compute ceiling (fp32) = {P_FP32} TFLOPS (measured)")
    
    # coalesced 工作点
    ax.scatter([AI_VADD], [P_COALESCED], color="#ea580c", zorder=6, s=95,
               marker="*", edgecolors="black", linewidths=0.6,
               label=f"coalesced (measured)")
    
    # linecross 工作点
    ax.scatter([AI_VADD], [P_LINECROSS], color="#7c3aed", zorder=6, s=95,
               marker="X", edgecolors="black", linewidths=0.6,
               label=f"linecross s=32 (measured)")
    
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel("Arithmetic Intensity (FLOP / Byte)", fontsize=11)
    ax.set_ylabel("Performance (TFLOPS)", fontsize=11)
    ax.set_title("Roofline: Radeon RX 9070 XT vector add", fontsize=12)
    ax.grid(True, which="both", ls=":", alpha=0.35)
    ax.legend(loc="upper left", fontsize=8, framealpha=0.92)
    ax.set_xlim(1e-2, 1e3)
    ax.set_ylim(1e-3, 40)
    
    plt.tight_layout()
    plt.show()
    
except ImportError:
    print("matplotlib 不可用，跳过绘图")

*（图示：第 5、6 章实测数据经绘图脚本生成的 9070XT Roofline 工作点。）*

两个工作点的区别在纵轴：

- **coalesced** 的有效带宽为 603 GB/s，点略高于 GDDR6 参考线；
- **linecross stride=32** 的有效带宽为 89.7 GB/s，点明显更低。

这张图只描述两个配置的实测结果，不单独解释 6.7 倍差距来自哪里。第 6 章已经看到，`linecross stride=32` 同时改变了地址排布、每线程循环次数和 Grid Size。

## 7.3 工作点离线很远怎么办

这一节给出一个入门排查顺序。不要一看到工作点低就立刻研究全部硬件细节，先从最容易验证的方向开始。

| 现象 | 第一个问题 | 最简单的检查 |
| ---- | ---- | ---- |
| AI 很低，离斜线远 | 地址是否连续 | 做连续 / 分散访存对照 |
| AI 很高，离水平线远 | 是否走了矩阵计算路径 | 对比 WMMA 和普通 VALU 实现 |
| 输入很小，点很低 | launch 是否占了大头 | 放大输入，观察时间是否近似线性增长 |
| 许多短 kernel 串联 | 是否频繁启动和同步 | 看 kernel trace 的数量与间隔 |
| 使用大量寄存器或 LDS | 是否限制了 occupancy | 对比 VGPR、SGPR、LDS 用量 |

回到 vector add：它的 AI 只有 0.083，所以先从访存方向检查是合理的。不过，第 6 章的 stride 还会同时改变每线程循环次数和 Grid Size；当前曲线只能提示方向，不能单独证明访存合并就是全部原因。下一步应补一个线程数和每线程工作量都固定的对照。

> **排查方法论**：先用 Roofline 选方向，再用 profiler 缩小范围，最后做一个只改一个变量的实验。

### 步骤7：解读工作点位置

根据工作点在 Roofline 图上的位置，选择排查方向：

In [ ]:
# 计算拐点（memory ceiling 和 compute ceiling 的交点）
BW_GDDR6 = 510  # GB/s
P_FP32 = 10.6   # TFLOPS
ridge_point = P_FP32 / (BW_GDDR6 / 1e3)  # FLOP/Byte

print(f"Roofline 拐点: AI = {ridge_point:.2f} FLOP/Byte")
print(f"\nvector add AI = {AI:.4f} FLOP/Byte")

if AI < ridge_point:
    print(f"  -> 位于拐点左侧，理论上是 memory-bound")
    print(f"  -> 下一步应先查访存：地址是否连续、是否有多余读写")
else:
    print(f"  -> 位于拐点右侧，理论上是 compute-bound")
    print(f"  -> 下一步应先查计算：是否走了矩阵计算路径、是否使用 WMMA")

# 计算工作点离 memory ceiling 的距离
print(f"\ncoalesced 工作点:")
print(f"  有效带宽 = {chapter6_results['coalesced']['effective_bw_gbps']:.1f} GB/s")
print(f"  memory ceiling = {BW_GDDR6} GB/s")
ratio_coalesced = chapter6_results['coalesced']['effective_bw_gbps'] / BW_GDDR6
print(f"  利用率 = {ratio_coalesced:.1%}")

print(f"\nlinecross stride=32 工作点:")
print(f"  有效带宽 = {chapter6_results['linecross_stride32']['effective_bw_gbps']:.1f} GB/s")
print(f"  memory ceiling = {BW_GDDR6} GB/s")
ratio_linecross = chapter6_results['linecross_stride32']['effective_bw_gbps'] / BW_GDDR6
print(f"  利用率 = {ratio_linecross:.1%}")
print(f"  -> 离 memory ceiling 很远，访存效率不高")

## 7.4 写一页性能记录

这一节把前面的结果记下来，目标是让未来的你知道：当时测了什么、结果怎样、为什么准备这样改。

一页记录保留五项就够：

| 项目 | 写什么 |
| ---- | ---- |
| 环境与对象 | OS、GPU、ROCm、kernel、输入规模和 dtype |
| 运行命令 | 能再次执行的 benchmark / profiling 命令 |
| 关键结果 | 时间口径（min / median）和两三个重要数字 |
| 当前判断 | 用一句话解释慢在哪里 |
| 下一步 | 只改一个变量的实验 |

可以直接使用下面这个简短模板：

```markdown
# 性能记录：<workload>

## 环境与对象
- OS / GPU / ROCm：<版本>
- kernel / 输入：<名称、shape、dtype>
- 计时口径：<warmup、repeat、min 或 median>

## 运行命令
<benchmark 命令>
<可选：profiling 命令>

## 关键结果
| 版本 | 时间 | 有效带宽或吞吐 |
| ---- | ----: | ----: |

## 当前判断
<哪一部分慢，依据是什么>

## 下一步
<只改哪个变量，准备观察什么>
```

套到本章的 vector add 上，核心结果只有三行：

| 版本 | 时间 | 有效带宽 |
| ---- | ----: | ----: |
| coalesced | 0.334 ms | 603 GB/s |
| linecross stride=32 | 2.25 ms | 89.7 GB/s |

当前判断可以写成：`linecross` 的时间会随 stride 整体增加，但 stride 同时改变了地址排布和工作划分。下一步应固定线程数与每线程循环次数，只打开或关闭数据预重排，再重新测量。

### 步骤8：生成性能记录

按照第7章模板，生成一页性能记录：

In [ ]:
performance_record = f"""
# 性能记录：vector add (coalesced vs linecross)

## 环境与对象
- OS / GPU / ROCm: Ubuntu 24.04 / Radeon RX 9070 XT (gfx1201) / ROCm 7.13
- kernel / 输入: vector_add / 16M float32
- 计时口径: warmup=20, repeat=100, min 和 median

## 运行命令
```bash
./vector_add_bench --kernel coalesced --size 16777216 --block 256 --warmup 20 --repeat 100
./vector_add_bench --kernel linecross --size 16777216 --block 256 --stride 32 --warmup 20 --repeat 100
```

## 关键结果
| 版本 | 时间 (ms) | 有效带宽 (GB/s) | AI (FLOP/Byte) | 性能 (TFLOPS) |
|------|-----------|----------------|----------------|---------------|
| coalesced | {chapter6_results['coalesced']['time_ms']:.3f} | {chapter6_results['coalesced']['effective_bw_gbps']:.1f} | {chapter6_results['coalesced']['ai']:.4f} | {chapter6_results['coalesced']['performance_tflops']:.4f} |
| linecross stride=32 | {chapter6_results['linecross_stride32']['time_ms']:.2f} | {chapter6_results['linecross_stride32']['effective_bw_gbps']:.1f} | {chapter6_results['linecross_stride32']['ai']:.4f} | {chapter6_results['linecross_stride32']['performance_tflops']:.4f} |

## 当前判断
- vector add AI = {AI:.4f} FLOP/Byte，位于 Roofline 拐点左侧，理论上是 memory-bound
- coalesced 有效带宽 {chapter6_results['coalesced']['effective_bw_gbps']:.1f} GB/s，接近 memory ceiling ({BW_GDDR6} GB/s)
- linecross stride=32 有效带宽 {chapter6_results['linecross_stride32']['effective_bw_gbps']:.1f} GB/s，离 memory ceiling 很远（利用率 {ratio_linecross:.1%}）
- stride 同时改变了地址排布、每线程循环次数和 Grid Size，这是组合效果

## 下一步
固定线程数与每线程循环次数，只打开或关闭数据预重排，再重新测量。
这样才能单独验证访存合并的影响。
"""

print(performance_record)

## 7.5 Part 1 的四步闭环

这一节把 Part 1 收成一条后面可以反复复用的路线。

*（图示：Part 1 建立的四步性能优化闭环——量准 → 找到慢点 → 解释 → 验证 → 进入下一版实现 → 量准……）*

这四步分别回答：

1. **量准**：这个数字能不能重复出现；
2. **找到慢点**：时间花在哪个 kernel；
3. **解释**：更像访存问题、计算问题，还是启动开销；
4. **验证**：只改一个变量，结果是否按预期变化。

Part 2 的 Reduction、Softmax、GEMM 和 Attention 会继续使用这条路线。算子会更复杂，但你仍然不需要一次看完所有工具输出，只要沿着当前问题一步步缩小范围。

## 本章小结

- Roofline 先用横轴和拐点判断理论瓶颈方向，再看工作点离对应上限还有多远。
- vector add 的 AI 约为 0.083 FLOP/Byte，理论上位于 memory-bound 一侧；有效带宽点还会受到算法口径和 cache 的影响。
- Roofline 负责选择排查方向，`rocprofv3` 和单变量实验负责找到更具体的原因。
- 一页性能记录只需要环境、命令、结果、判断和下一步。

## Expected Output / Interpretation

### 算术强度（arithmetic intensity）计算预期输出

```
vector add 算术强度 (arithmetic intensity):
  bytes_per_elem = 12 Byte
  flops_per_elem = 1 FLOP
  AI = 0.0833 FLOP/Byte

AI 很低（0.083），说明 vector add 是典型的 memory-bound 算子
```

### Roofline 图预期特征

在 9070XT (gfx1201) + ROCm 7.13 上，预期看到：

1. **蓝色斜线**（memory ceiling）：510 GB/s，来自第3章大数组 copy 实测
2. **红色水平线**（compute ceiling fp32）：10.6 TFLOPS，来自第3章 torch.matmul 实测
3. **橙色星形点**（coalesced）：AI=0.083, P≈0.050 TFLOPS，接近 memory ceiling
4. **紫色叉形点**（linecross stride=32）：AI=0.083, P≈0.0075 TFLOPS，明显更低

### 工作点位置解读

```
Roofline 拐点: AI = 20.78 FLOP/Byte

vector add AI = 0.0833 FLOP/Byte
  -> 位于拐点左侧，理论上是 memory-bound
  -> 下一步应先查访存：地址是否连续、是否有多余读写

coalesced 工作点:
  有效带宽 = 603.0 GB/s
  memory ceiling = 510 GB/s
  利用率 = 118.2%

linecross stride=32 工作点:
  有效带宽 = 89.7 GB/s
  memory ceiling = 510 GB/s
  利用率 = 17.6%
  -> 离 memory ceiling 很远，访存效率不高
```

**解读**：
- coalesced 的有效带宽（603 GB/s）略高于 memory ceiling（510 GB/s），这不表示突破硬件上限
- 有效带宽统计的是算法有效字节，工作集和写路径还可能受到 cache 影响
- linecross stride=32 利用率只有 17.6%，说明访存效率很低

### source attribution 说明

每条参考线都必须标注来源：
- **memory ceiling**: 第3章大数组 copy 实测 GDDR6 稳态带宽
- **compute ceiling**: 第3章 torch.matmul fp32 实测
- **不使用**: 理论峰值、其他架构的数字、未经验证的估算值

## Pass Criteria

本章操作通过标准：

### 必须满足

1. ✅ 能计算算术强度（AI = FLOP / Byte）
2. ✅ 理解 Roofline 的三个读图动作：横轴位置、纵轴高度、离上限距离
3. ✅ 能把第6章实测数据画到 Roofline 图上（或运行绘图脚本）
4. ✅ 理解 memory ceiling 和 compute ceiling 的来源（source attribution）
5. ✅ 能根据工作点位置选择排查方向（memory-bound vs compute-bound）

### 推荐完成

6. ⭐ 运行 `plot_roofline_ch6.py`，生成完整的 Roofline 图
7. ⭐ 计算工作点离 memory ceiling 的利用率
8. ⭐ 生成一页性能记录，包含环境、命令、结果、判断和下一步

### 常见问题排查

| 现象 | 可能原因 | 解决方法 |
|------|----------|----------|
| matplotlib 导入失败 | 缺少依赖 | `pip install matplotlib numpy` |
| 图片不显示 | 脚本未生成 PNG | 检查 `--save` 参数和输出路径 |
| 有效带宽 > memory ceiling | 算法口径 vs 硬件口径 | 正常现象，cache 影响 |
| 拐点位置不对 | 参考线数值错误 | 检查 source attribution |
| 工作点位置偏移 | AI 或性能计算错误 | 重新核对 bytes / ops 口径 |

---

## 延伸阅读

- [Roofline Model 原论文](https://dl.acm.org/doi/10.1145/1498765.1498785)
- [ROCm Profiling Tools 总览](https://rocm.docs.amd.com/en/latest/conceptual/gpu-arch/rocm-tools.html)
- [HIP Performance Guidelines](https://rocm.docs.amd.com/projects/HIP/en/latest/how-to/performance_guidelines.html)

**Part 1 完成！** 下一步进入 Part 2 — Reduction、Softmax、GEMM 和 Attention 的 kernel 实现。